# 04 BM25

## Objetivos

* Implementar BM25 y comparar su ranking con el de TF-IDF + coseno sobre la misma colección de documentos.
* Calcular el aporte de cada término en BM25, identificar qué controlan $k_1$ y $b$, y justificar un cambio de posición usando las estadísticas de los documentos.

**1. Carga el corpus de Project Gutenberg**

Utilizar el corpus de Project Gutenberg y el preprocesamiento de la práctica anterior.

Recuperar una tabla con los campos **Identificador de documento | Título | Texto**

Calcule la columna `tokens` para la tabla, con lista de tokens normalizados de cada documento.

Calcula DF e IDF.

In [ ]:
import re, math, time, unicodedata, random
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 120)

PALABRA = re.compile(r"[a-z0-9]+(?:'[a-z0-9]+)?")

def normalizar(texto):
    texto = unicodedata.normalize("NFKD", texto.lower().replace("\u2019", "'"))
    return texto.encode("ascii", "ignore").decode("ascii")

def tokenizar(texto):
    return PALABRA.findall(normalizar(texto))

def recortar(texto):
    ini = re.search(r"\*{3}\s*START OF (?:THE|THIS) PROJECT GUTENBERG[^\n]*\n", texto, re.I)
    if ini:
        texto = texto[ini.end():]
    fin = re.search(r"\*{3}\s*END OF (?:THE|THIS) PROJECT GUTENBERG", texto, re.I)
    if fin:
        texto = texto[:fin.start()]
    return texto

RUTA_CORPUS = None  

def buscar_carpeta_epub():
    if RUTA_CORPUS:
        return Path(RUTA_CORPUS)
    casa = Path.home()
    for base in (casa / "Downloads", casa / "Descargas", casa / "Desktop", Path.cwd()):
        for sub in (("txt-files", "txt-files", "cache", "epub"),
                    ("txt-files", "cache", "epub"),
                    ("cache", "epub")):
            ruta = base.joinpath(*sub)
            if ruta.is_dir():
                return ruta
    raise FileNotFoundError("No se encontró la carpeta cache/epub, define RUTA_CORPUS")

def elegir_txt(carpeta):
    candidatos = [f for f in carpeta.glob("*.txt") if f.stat().st_size > 0]
    if not candidatos:
        return None
    principal = carpeta / f"pg{carpeta.name}.txt"
    if principal in candidatos:
        return principal
    return min(candidatos, key=lambda f: len(f.name))

def leer_texto(ruta):
    codificacion = "latin-1" if ruta.name.lower().endswith("-8.txt") else "utf-8"
    return ruta.read_bytes().decode(codificacion, errors="ignore")

def extraer_titulo(texto, id_libro):
    m = re.search(r"^\ufeff?Title:[ \t]*(.+(?:\n[ \t]+\S.*)*)", texto[:6000], re.M)
    if m:
        return re.sub(r"\s+", " ", m.group(1)).strip()
    return f"pg{id_libro}"

N_DOCS, SEMILLA = 1000, 2026

CARPETA_EPUB = buscar_carpeta_epub()
carpetas = sorted((c for c in CARPETA_EPUB.iterdir() if c.is_dir() and c.name.isdigit()),
                  key=lambda c: int(c.name))
libros = {}
for carpeta in carpetas:
    archivo = elegir_txt(carpeta)
    if archivo is not None:
        libros[int(carpeta.name)] = archivo
print("Documentos válidos en Gutenberg:", len(libros))

muestra = random.Random(SEMILLA).sample(sorted(libros), min(N_DOCS, len(libros)))


unicas = {}   
t0 = time.perf_counter()
for k, id_libro in enumerate(muestra, 1):
    texto_crudo = leer_texto(libros[id_libro])
    texto = recortar(texto_crudo)
    filas.append({
        "doc_id": str(id_libro),                   
        "titulo": extraer_titulo(texto_crudo, id_libro),
        "texto": texto,
        "tokens": [unicas.setdefault(p, p) for p in tokenizar(texto)],
    })
    if k % 100 == 0:
        print(f"{k} de {len(muestra)} libros procesados")
print(f"Carga y tokenización: {time.perf_counter() - t0:.1f} s")

df_docs = (pd.DataFrame(filas)
             .sort_values("doc_id", key=lambda s: s.astype(int))  
             .reset_index(drop=True))
df_docs["longitud"] = df_docs["tokens"].str.len()

# ---------- DF e IDF ----------
N = len(df_docs)
df_term = Counter()
for toks in df_docs["tokens"]:
    df_term.update(set(toks))          

# IDF para BM25 (Robertson–Spärck Jones, versión no negativa)
idf_bm25 = {t: math.log(1 + (N - d + 0.5) / (d + 0.5)) for t, d in df_term.items()}
# IDF de TF-IDF (el mismo del 03, para el coseno)
idf_tfidf = {t: math.log(N / d) + 1 for t, d in df_term.items()}

avgdl = df_docs["longitud"].mean()

print(f"\nNúmero de documentos: {N}")
print(f"Tamaño del vocabulario: {len(df_term):,}")
print(f"Longitud media de documento (avgdl): {avgdl:,.1f} tokens")
display(df_docs[["doc_id", "titulo", "longitud"]].head(10))

# Verificación rápida de DF/IDF con un término cualquiera
t_ej = "captain"
print(f"\n'{t_ej}': DF={df_term[t_ej]} | IDF_BM25={idf_bm25[t_ej]:.4f} | IDF_TFIDF={idf_tfidf[t_ej]:.4f}")

Documentos válidos en Gutenberg: 78145
100 de 1000 libros procesados
200 de 1000 libros procesados
300 de 1000 libros procesados
400 de 1000 libros procesados
500 de 1000 libros procesados
600 de 1000 libros procesados
700 de 1000 libros procesados
800 de 1000 libros procesados
900 de 1000 libros procesados
1000 de 1000 libros procesados
Carga y tokenización: 43.1 s

Número de documentos: 1000
Tamaño del vocabulario: 953,264
Longitud media de documento (avgdl): 60,739.6 tokens


,doc_id,titulo,longitud
0,29,The 1990 United States Census [incomplete],27351
1,290,The Stark Munro Letters,78183
2,361,Miss Billy — Married,77677
3,680,The Golden Threshold,8244
4,709,The Princess and Curdie,56407
5,733,History of the Decline and Fall of the Roman Empire — Volume 3,247663
6,795,"Ballads and Lyrics of Old France, with Other Poems",14741
7,997,Divina Commedia di Dante: Inferno,32800
8,1041,Shakespeare's Sonnets,17764
9,1313,Over the Sliprails,44867



'captain': DF=445 | IDF_BM25=0.8096 | IDF_TFIDF=1.8097


**2. Benchmarking**

Seleccione **ocho consultas en el idioma del corpus** y registra en una línea qué busca el usuario con cada una. Incluye términos muy frecuentes, términos poco frecuentes, y consultas de distintas longitudes. Usa $df$ para justificar las elecciones.

Para **TF-IDF + coseno** conserva la representación dispersa de la matriz.

Para **BM25** define $k_1=1.2$, $b=0.75$. Mantén intactas las estadísticas de la colección al cambiar de consulta.

Para cada consulta, ordena por puntuación descendente y desempata por `doc_id` ascendente (trata los IDs como cadenas en ambos métodos). Presenta hasta diez documentos con score positivo por método. Una consulta sin términos en el vocabulario produce dos listas vacías.

In [2]:
candidatos = [
    # muy frecuentes
    "the", "man", "captain", "king", "love", "war", "sea", "house",
    # frecuencia media
    "ship", "storm", "detective", "emperor", "bread", "garden",
    # poco frecuentes
    "legions", "voyage", "murder", "oven", "harpoon", "sonnet", "altimeter", "cathedral",
]

tabla_df = pd.DataFrame({
    "término": candidatos,
    "df": [df_term.get(t, 0) for t in candidatos],
    "% docs": [round(100 * df_term.get(t, 0) / N, 1) for t in candidatos],
    "IDF_BM25": [round(idf_bm25.get(t, 0), 3) for t in candidatos],
}).sort_values("df", ascending=False).reset_index(drop=True)

display(tabla_df)

,término,df,% docs,IDF_BM25
0,the,858,85.8,0.154
1,man,814,81.4,0.206
2,house,728,72.8,0.318
3,love,670,67.0,0.401
4,sea,641,64.1,0.445
5,war,638,63.8,0.450
6,king,553,55.3,0.592
7,garden,527,52.7,0.641
8,storm,510,51.0,0.673
9,bread,506,50.6,0.681


In [ ]:
from array import array
from scipy import sparse

# ---------- Consultas ----------
consultas = {
    "Q1": {"texto": "altimeter",                               "necesidad": "Instrumentos de altitud y aviación"},
    "Q2": {"texto": "love",                                    "necesidad": "Historias de amor"},
    "Q3": {"texto": "the man house love",                      "necesidad": "Relatos domésticos o sentimentales"},
    "Q4": {"texto": "sonnet love",                             "necesidad": "Poesía amorosa en forma de soneto"},
    "Q5": {"texto": "murder detective",                        "necesidad": "Novelas policiales"},
    "Q6": {"texto": "bread oven",                              "necesidad": "Recetas de panadería"},
    "Q7": {"texto": "roman emperor legions war",               "necesidad": "Historia militar de Roma"},
    "Q8": {"texto": "storm captain voyage ship sea harpoon",   "necesidad": "Aventuras marítimas"},
}


vocab = sorted(df_term)
pos = {t: j for j, t in enumerate(vocab)}
V = len(vocab)
doc_ids = df_docs["doc_id"].tolist()                 # cadenas
dl = df_docs["longitud"].to_numpy(dtype=np.float64)  # |d|

indptr, indices, datos = array("q", [0]), array("i"), array("d")
for toks in df_docs["tokens"]:
    fila = sorted((pos[t], f) for t, f in Counter(toks).items())
    indices.extend(j for j, _ in fila)
    datos.extend(f for _, f in fila)
    indptr.append(len(indices))

TF = sparse.csr_matrix((np.frombuffer(datos, dtype=np.float64),
                        np.frombuffer(indices, dtype=np.int32),
                        np.frombuffer(indptr, dtype=np.int64)), shape=(N, V))
TFc = TF.tocsc()                                     # acceso rápido por término (para BM25)

idf_tfidf_vec = np.array([idf_tfidf[t] for t in vocab])
idf_bm25_vec  = np.array([idf_bm25[t] for t in vocab])


W = (TF @ sparse.diags(idf_tfidf_vec)).tocsr()
normas_docs = np.sqrt(W.multiply(W).sum(axis=1)).A1

def terminos_consulta(texto):
    toks = tokenizar(texto)
    return [t for t in toks if t in pos], [t for t in toks if t not in pos]

def scores_tfidf(texto):
    en_vocab, _ = terminos_consulta(texto)
    if not en_vocab:
        return np.zeros(N)
    tf_q = Counter(en_vocab)
    cols = sorted(pos[t] for t in tf_q)
    vals = [tf_q[vocab[j]] * idf_tfidf_vec[j] for j in cols]
    q = sparse.csr_matrix((vals, ([0]*len(cols), cols)), shape=(1, V))
    punto = (W @ q.T).toarray().ravel()
    nq = np.sqrt((q.multiply(q)).sum())
    den = nq * normas_docs
    return np.divide(punto, den, out=np.zeros_like(punto), where=den > 0)


def scores_bm25(texto, k1=1.2, b=0.75):
    en_vocab, _ = terminos_consulta(texto)
    sc = np.zeros(N)
    for t in set(en_vocab):                          # término repetido en la consulta cuenta una vez
        j = pos[t]
        col = TFc[:, j]
        filas, f = col.indices, col.data
        denom = f + k1 * (1 - b + b * dl[filas] / avgdl)
        sc[filas] += idf_bm25_vec[j] * f * (k1 + 1) / denom
    return sc

def aporte_bm25(termino, doc_id, k1=1.2, b=0.75):
    """Aporte BM25 de un término a un documento (lo usarás en los pasos 3 y 4)."""
    i, j = doc_ids.index(doc_id), pos[termino]
    f = TF[i, j]
    return idf_bm25_vec[j] * f * (k1 + 1) / (f + k1 * (1 - b + b * dl[i] / avgdl)) if f else 0.0


def top_k(scores, k=10):
    idx = np.nonzero(scores > 0)[0]
    orden = sorted(idx, key=lambda i: (-scores[i], doc_ids[i]))
    return orden[:k]

filas_info = []
for qid, c in consultas.items():
    en_vocab, fuera = terminos_consulta(c["texto"])
    filas_info.append({
        "Consulta": qid, "Texto": c["texto"], "Nº términos": len(en_vocab),
        "df por término": {t: df_term[t] for t in en_vocab},
        "Fuera de vocabulario": fuera,
        "Qué busca": c["necesidad"],
    })
display(pd.DataFrame(filas_info))

resultados = {}
for qid, c in consultas.items():
    s_tf, s_bm = scores_tfidf(c["texto"]), scores_bm25(c["texto"])
    resultados[qid] = {"tfidf": top_k(s_tf), "bm25": top_k(s_bm), "s_tfidf": s_tf, "s_bm25": s_bm}
    print(f"{qid} '{c['texto']}': TF-IDF devuelve {len(resultados[qid]['tfidf'])} docs | "
          f"BM25 devuelve {len(resultados[qid]['bm25'])} docs")

# Prueba del caso borde: consulta sin términos del vocabulario -> dos listas vacías
prueba = "zzzxqv"
print("\nConsulta fuera de vocabulario:", top_k(scores_tfidf(prueba)), top_k(scores_bm25(prueba)))

,Consulta,Texto,Nº términos,df por término,Fuera de vocabulario,Qué busca
0,Q1,altimeter,1,{'altimeter': 6},[],Instrumentos de altitud y aviación
1,Q2,love,1,{'love': 670},[],Historias de amor
2,Q3,the man house love,4,"{'the': 858, 'man': 814, 'house': 728, 'love': 670}",[],Relatos domésticos o sentimentales
3,Q4,sonnet love,2,"{'sonnet': 67, 'love': 670}",[],Poesía amorosa en forma de soneto
4,Q5,murder detective,2,"{'murder': 363, 'detective': 91}",[],Novelas policiales
5,Q6,bread oven,2,"{'bread': 506, 'oven': 184}",[],Recetas de panadería
6,Q7,roman emperor legions war,4,"{'roman': 412, 'emperor': 242, 'legions': 94, 'war': 638}",[],Historia militar de Roma
7,Q8,storm captain voyage ship sea harpoon,6,"{'storm': 510, 'captain': 445, 'voyage': 370, 'ship': 457, 'sea': 641, 'harpoon': 25}",[],Aventuras marítimas


Q1 'altimeter': TF-IDF devuelve 6 docs | BM25 devuelve 6 docs
Q2 'love': TF-IDF devuelve 10 docs | BM25 devuelve 10 docs
Q3 'the man house love': TF-IDF devuelve 10 docs | BM25 devuelve 10 docs
Q4 'sonnet love': TF-IDF devuelve 10 docs | BM25 devuelve 10 docs
Q5 'murder detective': TF-IDF devuelve 10 docs | BM25 devuelve 10 docs
Q6 'bread oven': TF-IDF devuelve 10 docs | BM25 devuelve 10 docs
Q7 'roman emperor legions war': TF-IDF devuelve 10 docs | BM25 devuelve 10 docs
Q8 'storm captain voyage ship sea harpoon': TF-IDF devuelve 10 docs | BM25 devuelve 10 docs

Consulta fuera de vocabulario: [] []


**3. Compara los rankings**

Muestra por consulta una tabla como esta. Usa los IDs para comparar.

| Posición | ID y título TF-IDF | Coseno | ID y título BM25 | Score BM25 |
|---:|---|---:|---|---:|
| 1 | … | … | … | … |

Calcula cuántos IDs comparten los dos top 10 con $|T_{\mathrm{TFIDF}}\cap T_{\mathrm{BM25}}|$. Reporta también el tamaño real de cada lista y el primer documento de cada método. Dos listas pueden tener muchos IDs en común y ordenarlos de forma distinta.

Elige una consulta y dos documentos que cambien de posición o cuya distancia en el ranking cambie. Registra $|d|$, $f(t,d)$, $df(t)$ y el aporte BM25 de cada término de consulta para ambos documentos. Relaciona esas cifras con la saturación, la longitud o la rareza del término. Usa los scores de cada método para explicar su propio ranking.

In [4]:
def corto(i, n=45):
    t = df_docs["titulo"][i]
    return f"{doc_ids[i]} · {t[:n]}{'…' if len(t) > n else ''}"

def tabla_comparativa(qid):
    r = resultados[qid]
    tf, bm = r["tfidf"], r["bm25"]
    filas = []
    for p in range(max(len(tf), len(bm))):
        filas.append({
            "Posición": p + 1,
            "ID y título TF-IDF": corto(tf[p]) if p < len(tf) else "",
            "Coseno": round(float(r["s_tfidf"][tf[p]]), 6) if p < len(tf) else np.nan,
            "ID y título BM25": corto(bm[p]) if p < len(bm) else "",
            "Score BM25": round(float(r["s_bm25"][bm[p]]), 4) if p < len(bm) else np.nan,
        })
    return pd.DataFrame(filas)

resumen = []
for qid, c in consultas.items():
    r = resultados[qid]
    T_tf = {doc_ids[i] for i in r["tfidf"]}
    T_bm = {doc_ids[i] for i in r["bm25"]}
    print(f"\n{qid}: '{c['texto']}'  |  {c['necesidad']}")
    print(f"|T_TFIDF ∩ T_BM25| = {len(T_tf & T_bm)}  "
          f"(tamaño TF-IDF = {len(T_tf)}, tamaño BM25 = {len(T_bm)})")
    display(tabla_comparativa(qid))
    resumen.append({
        "Consulta": qid,
        "Tamaño TF-IDF": len(T_tf), "Tamaño BM25": len(T_bm),
        "Intersección": len(T_tf & T_bm),
        "Primero TF-IDF": corto(r["tfidf"][0], 30) if r["tfidf"] else "",
        "Primero BM25": corto(r["bm25"][0], 30) if r["bm25"] else "",
        "Mismo primero": bool(r["tfidf"] and r["bm25"] and r["tfidf"][0] == r["bm25"][0]),
    })

print("\n=== Resumen ===")
display(pd.DataFrame(resumen))


Q1: 'altimeter'  |  Instrumentos de altitud y aviación
|T_TFIDF ∩ T_BM25| = 6  (tamaño TF-IDF = 6, tamaño BM25 = 6)


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,73145 · The thought-feeders,0.010208,79582 · Named right,9.1764
1,2,79582 · Named right,0.009046,73145 · The thought-feeders,8.1757
2,3,"39488 · The Airship ""Golden Hind""",0.002117,"39488 · The Airship ""Golden Hind""",7.6156
3,4,32530 · Armageddon—2419 A.D.,0.001721,32530 · Armageddon—2419 A.D.,6.4750
4,5,"44862 · Linda Carlton, Air Pilot",0.001115,"44862 · Linda Carlton, Air Pilot",5.5760
5,6,75969 · Scissors A novel of youth,0.000513,75969 · Scissors A novel of youth,3.9194



Q2: 'love'  |  Historias de amor
|T_TFIDF ∩ T_BM25| = 7  (tamaño TF-IDF = 10, tamaño BM25 = 10)


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,1041 · Shakespeare's Sonnets,0.126636,63294 · Thoughts of Saint Thérèse of the Child Jesus,0.8789
1,2,63294 · Thoughts of Saint Thérèse of the Child Jesus,0.089821,1041 · Shakespeare's Sonnets,0.8787
2,3,"795 · Ballads and Lyrics of Old France, with Other …",0.087098,19284 · Paris Talks,0.8780
3,4,"75479 · Willow's forge, and other poems",0.075888,2491 · Love or Fame; and Other Poems,0.8778
4,5,2491 · Love or Fame; and Other Poems,0.072183,"795 · Ballads and Lyrics of Old France, with Other …",0.8775
5,6,36584 · The Blind Beggar of Jericho,0.069682,29557 · The Lord of Glory,0.8775
6,7,10518 · Poems,0.059390,10518 · Poems,0.8768
7,8,19284 · Paris Talks,0.051501,66405 · Annette and Sylvie: Being Volume One of The S…,0.8758
8,9,"65133 · Chambers's Journal of Popular Literature, Sci…",0.049972,"33519 · The Nest, The White Pagoda, The Suicide, A Fo…",0.8755
9,10,29557 · The Lord of Glory,0.046984,"36071 · Hesperus; or, Forty-Five Dog-Post-Days: A Bio…",0.8752



Q3: 'the man house love'  |  Relatos domésticos o sentimentales
|T_TFIDF ∩ T_BM25| = 0  (tamaño TF-IDF = 10, tamaño BM25 = 10)


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,52581 · Fountains Abbey: The story of a mediæval mona…,0.371809,26002 · Linda Tressel,2.3538
1,2,19498 · Banzai! by Parabellum,0.366432,36666 · The Sins of the Father: A Romance of the Sout…,2.3503
2,3,73323 · Soissons before and during the war,0.366145,37416 · The Princess of Bagdad: A Play In Three Acts,2.3494
3,4,"61475 · Byzantine Constantinople, the walls of the ci…",0.360881,1970 · A Poor Wise Man,2.3493
4,5,"56880 · The Secret Doctrine, Vol. 3 of 4",0.356526,"36071 · Hesperus; or, Forty-Five Dog-Post-Days: A Bio…",2.3477
5,6,19284 · Paris Talks,0.355277,10379 · At love's cost,2.3466
6,7,72694 · The Christmas city Bethlehem across the ages,0.354561,62883 · The Old Room,2.3464
7,8,48013 · Forty Thousand Miles Over Land and Water,0.352462,12354 · Pink and White Tyranny,2.3463
8,9,65707 · The Crime of Caste in Our Country,0.351977,55091 · The Book of the Thousand Nights and a Night —…,2.3463
9,10,5185 · Kalevala : the Epic Poem of Finland — Volume …,0.351388,30247 · Mabel's Mistake,2.3457


... (8 outputs left out)


Q8: 'storm captain voyage ship sea harpoon'  |  Aventuras marítimas
|T_TFIDF ∩ T_BM25| = 3  (tamaño TF-IDF = 10, tamaño BM25 = 10)


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,73145 · The thought-feeders,0.052084,"73758 · Lake Ngami or, Explorations and discoveries d…",13.7801
1,2,61497 · The Eagles Gather,0.045240,"64999 · The Golden Harpoon; Or, Lost Among the Floes:…",13.7701
2,3,"64999 · The Golden Harpoon; Or, Lost Among the Floes:…",0.037528,4534 · Sylvia's Lovers — Volume 1,13.7302
3,4,74528 · An essay on the preservation of shipwrecked m…,0.037045,51000 · School Reading By Grades: Fifth Year,13.7231
4,5,74551 · David goes voyaging,0.031324,74551 · David goes voyaging,13.2600
5,6,41399 · Admirals of the British Navy,0.028976,75370 · David goes to Greenland,13.0713
6,7,"33282 · The Boy Pilot of the Lakes; Or, Nat Morton's …",0.027993,"69994 · The triumphs of perseverance and enterprise, …",12.9116
7,8,73464 · Sailing ships the story of their development …,0.027906,"72861 · Head-hunters, black, white, and brown",12.8264
8,9,27520 · A Sailor of King George,0.026669,37669 · In the Arctic Seas,12.1658
9,10,23374 · The Dingo Boys: The Squatters of Wallaby Rang…,0.026233,73464 · Sailing ships the story of their development …,11.6175



=== Resumen ===


,Consulta,Tamaño TF-IDF,Tamaño BM25,Intersección,Primero TF-IDF,Primero BM25,Mismo primero
0,Q1,6,6,6,73145 · The thought-feeders,79582 · Named right,False
1,Q2,10,10,7,1041 · Shakespeare's Sonnets,63294 · Thoughts of Saint Thérèse of t…,False
2,Q3,10,10,0,52581 · Fountains Abbey: The story of …,26002 · Linda Tressel,False
3,Q4,10,10,3,25574 · Recollections of Dante Gabriel…,25574 · Recollections of Dante Gabriel…,True
4,Q5,10,10,6,29331 · The Crevice,14818 · The Daughter of Anderson Crow,False
5,Q6,10,10,5,42631 · Friendship Club Cook Book,42631 · Friendship Club Cook Book,True
6,Q7,10,10,5,3574 · Recollections of the Private L…,44083 · The Count of the Saxon Shore; …,False
7,Q8,10,10,3,73145 · The thought-feeders,"73758 · Lake Ngami or, Explorations an…",False


**Movimientos entre rankings.** `Cambio = Pos TF-IDF − Pos BM25`: un valor positivo indica que el documento sube en BM25. La tabla solo incluye documentos presentes en ambos top 10 y sirve para elegir la consulta y los dos documentos del análisis (se eligió Q7: 733 y 75682).

In [5]:
def movimientos(qid):
    r = resultados[qid]
    pos_tf = {doc_ids[i]: p + 1 for p, i in enumerate(r["tfidf"])}
    pos_bm = {doc_ids[i]: p + 1 for p, i in enumerate(r["bm25"])}
    comunes = set(pos_tf) & set(pos_bm)
    return [{
        "Consulta": qid, "doc_id": d,
        "Pos TF-IDF": pos_tf[d], "Pos BM25": pos_bm[d],
        "Cambio": pos_tf[d] - pos_bm[d],
        "|d|": int(dl[doc_ids.index(d)]),
    } for d in comunes]

filas_mov = [f for q in consultas for f in movimientos(q)]
mov = pd.DataFrame(filas_mov)
mov = mov.sort_values("Cambio", key=abs, ascending=False).reset_index(drop=True)
display(mov.head(15))

,Consulta,doc_id,Pos TF-IDF,Pos BM25,Cambio,|d|
0,Q7,733,10,3,7,247663
1,Q7,44083,8,1,7,71403
2,Q5,77807,9,2,7,42227
3,Q4,2491,5,10,-5,26233
4,Q2,19284,8,3,5,43236
5,Q4,795,3,7,-4,14741
6,Q6,79407,8,4,4,38470
7,Q2,29557,10,6,4,44220
8,Q5,76632,4,7,-3,20915
9,Q7,34397,5,2,3,33015


In [6]:
def analisis(qid, docs):
    texto = consultas[qid]["texto"]
    en_vocab, _ = terminos_consulta(texto)
    tf_q = Counter(en_vocab)
    nq = math.sqrt(sum((tf_q[t] * idf_tfidf[t]) ** 2 for t in tf_q))
    r = resultados[qid]
    pos_tf = {doc_ids[i]: p + 1 for p, i in enumerate(r["tfidf"])}
    pos_bm = {doc_ids[i]: p + 1 for p, i in enumerate(r["bm25"])}

    for d in docs:
        i = doc_ids.index(d)
        print(f"\nDoc {d} · {df_docs['titulo'][i][:60]}")
        print(f"|d| = {int(dl[i]):,}  |  avgdl = {avgdl:,.0f}  |  |d|/avgdl = {dl[i]/avgdl:.2f}"
              f"  |  Pos TF-IDF = {pos_tf.get(d, '>10')}  |  Pos BM25 = {pos_bm.get(d, '>10')}")
        filas = []
        for t in sorted(tf_q):
            f = int(TF[i, pos[t]])
            q_i = tf_q[t] * idf_tfidf[t]
            d_i = f * idf_tfidf[t]
            filas.append({
                "término": t, "f(t,d)": f, "df(t)": df_term[t],
                "IDF_BM25": round(idf_bm25[t], 3),
                "aporte BM25": round(aporte_bm25(t, d), 4),
                "aporte coseno": round(q_i * d_i / (nq * normas_docs[i]), 6),
            })
        tabla = pd.DataFrame(filas)
        display(tabla)
        print(f"Score BM25 = {tabla['aporte BM25'].sum():.4f}  |  Coseno = {tabla['aporte coseno'].sum():.6f}"
              f"  |  ||d|| (TF-IDF) = {normas_docs[i]:,.1f}")

analisis("Q7", ["733", "75682"])


Doc 733 · History of the Decline and Fall of the Roman Empire — Volume
|d| = 247,663  |  avgdl = 60,740  |  |d|/avgdl = 4.08  |  Pos TF-IDF = 10  |  Pos BM25 = 3


,término,"f(t,d)",df(t),IDF_BM25,aporte BM25,aporte coseno
0,emperor,361,242,1.418,3.0851,0.010712
1,legions,32,94,2.360,4.6193,0.001837
2,roman,355,412,0.887,1.9288,0.006409
3,war,231,638,0.450,0.9725,0.002461


Score BM25 = 10.6057  |  Coseno = 0.021419  |  ||d|| (TF-IDF) = 41,266.9

Doc 75682 · Waterloo
|d| = 27,628  |  avgdl = 60,740  |  |d|/avgdl = 0.45  |  Pos TF-IDF = 3  |  Pos BM25 = 5


,término,"f(t,d)",df(t),IDF_BM25,aporte BM25,aporte coseno
0,emperor,114,242,1.418,3.0998,0.034481
1,legions,4,94,2.360,4.4102,0.002341
2,roman,3,412,0.887,1.5774,0.000552
3,war,11,638,0.450,0.9293,0.001195


Score BM25 = 10.0167  |  Coseno = 0.038569  |  ||d|| (TF-IDF) = 4,048.2


In [7]:
# Verificación de Q3: un documento por método (ajusta los IDs si cambia el corpus)
analisis("Q3", ["52581", "26002"])


Doc 52581 · Fountains Abbey: The story of a mediæval monastery
|d| = 22,446  |  avgdl = 60,740  |  |d|/avgdl = 0.37  |  Pos TF-IDF = 1  |  Pos BM25 = >10


,término,"f(t,d)",df(t),IDF_BM25,aporte BM25,aporte coseno
0,house,81,728,0.318,0.6937,0.015058
1,love,2,670,0.401,0.6698,0.000420
2,man,20,814,0.206,0.4397,0.003115
3,the,2480,858,0.154,0.3378,0.353216


Score BM25 = 2.1410  |  Coseno = 0.371809  |  ||d|| (TF-IDF) = 3,667.5

Doc 26002 · Linda Tressel
|d| = 71,522  |  avgdl = 60,740  |  |d|/avgdl = 1.18  |  Pos TF-IDF = >10  |  Pos BM25 = 1


,término,"f(t,d)",df(t),IDF_BM25,aporte BM25,aporte coseno
0,house,194,728,0.318,0.6942,0.013129
1,love,111,670,0.401,0.8709,0.008489
2,man,233,814,0.206,0.4510,0.013209
3,the,2843,858,0.154,0.3377,0.147403


Score BM25 = 2.3538  |  Coseno = 0.182230  |  ||d|| (TF-IDF) = 10,074.7


**Observaciones por consulta.**

- **Q1 (`altimeter`, df=6):** ambos métodos devuelven los mismos 6 documentos, porque solo ellos contienen el término. Solo cambia el orden: BM25 pone primero a 79582 (9.1764) y TF-IDF a 73145 (0.010208). Los resultados son novelas que mencionan el término, no textos técnicos, así que el vocabulario coincide pero la necesidad de información se cumple solo en parte.
- **Q2 (`love`, df=670):** el score BM25 del top 10 va de 0.8789 a 0.8752. Con un solo término de IDF bajo (0.401) y la frecuencia saturada, casi solo la longitud separa a los documentos, y el desempate por `doc_id` pesa mucho. El coseno separa más (0.1266 a 0.0470).
- **Q3 (`the man house love`):** la intersección es 0. Los cuatro términos son muy comunes (df de 670 a 858; IDF_BM25 de 0.154 a 0.401) y casi todos los documentos los contienen, así que BM25 queda con scores casi empatados (2.3538 a 2.3457). Ninguno de los dos rankings es claramente relevante para la necesidad planteada. La celda de verificación de Q3 (más abajo) muestra longitudes y aportes de un par de documentos.
- **Q4 y Q6** son las consultas donde ambos métodos coinciden en el primer documento; sus intersecciones son 3 y 5 documentos.

**Convención de la tabla de movimientos:** `Cambio = Pos TF-IDF − Pos BM25`. Un valor positivo significa que el documento **sube** en BM25 respecto al coseno.

**Análisis de Q7 (docs 733 y 75682).**

| | 733 (Decline and Fall) | 75682 (Waterloo) |
|---|---:|---:|
| \|d\| | 247,663 (4.08·avgdl) | 27,628 (0.45·avgdl) |
| Pos. TF-IDF / BM25 | 10 / 3 | 3 / 5 |
| Score BM25 | 10.6057 | 10.0167 |
| Coseno | 0.021419 | 0.038569 |

- **Saturación (BM25):** el aporte por término tiene techo (k1+1)·IDF. En `emperor` el techo es 3.119: Waterloo (f=114) aporta 3.0998 y 733 (f=361, tres veces más) aporta 3.0851. Triplicar la frecuencia no mejora el score.
- **Longitud (BM25):** 733 es 4 veces más largo que la media, así que su denominador crece y su aporte baja. Waterloo, más corto que la media, compensa su menor f.
- **Por qué 733 queda por encima en BM25:** gana en `roman` (1.9288 frente a 1.5774) y en `legions` (4.6193 frente a 4.4102), donde Waterloo aún no satura (f=3 y f=4). Esa ventaja decide el orden.
- **Rareza:** `legions` (df=94, IDF 2.360) es el término de mayor aporte en ambos (4.6 y 4.4). `war` (df=638, IDF 0.450) aporta menos de 1.
- **Coseno:** usa TF crudo y divide entre ||d||, que vale 41,267 para 733 y 4,048 para Waterloo. En `emperor` el aporte de Waterloo (0.0345) es 3 veces el de 733 (0.0107) aunque 733 tenga más apariciones, porque su vector está mucho más concentrado en ese término.

**Conclusión:** el coseno con TF crudo no satura la frecuencia y su norma crece con la longitud, así que el libro largo cae al puesto 10. BM25 satura la frecuencia y corrige la longitud de forma acotada, y por eso 733 sube al puesto 3.

**4. Identifica el efecto de los parámetros**

Mantén fijos la colección y el preprocesamiento. Escoge una consulta con coincidencias en documentos de distintas longitudes y frecuencias.

| Ejecución | $k_1$ | $b$ | Qué examinar |
|---|---:|---:|---|
| Base | 1.2 | 0.75 | Posiciones y aportes de referencia |
| Sin ajuste por longitud | 1.2 | 0.00 | Cambios en documentos más cortos o más largos que la media |
| Mayor efecto de las repeticiones | 2.0 | 0.75 | Cambios en el aporte de términos con frecuencias distintas |

Muestra las posiciones de los mismos documentos en las tres configuraciones. Si no se mueven, compara sus aportes por término y explica por qué el cambio de puntuación no alteró su orden.

In [8]:
configs = {
    "Base":              (1.2, 0.75),
    "Sin longitud":      (1.2, 0.00),
    "Más repeticiones":  (2.0, 0.75),
}
qid4, texto4 = "Q7", consultas["Q7"]["texto"]

pos_cfg, sc_cfg = {}, {}
for nombre, (k1, b) in configs.items():
    s = scores_bm25(texto4, k1, b)
    orden = top_k(s, N)                                  # ranking completo con score > 0
    pos_cfg[nombre] = {doc_ids[i]: p + 1 for p, i in enumerate(orden)}
    sc_cfg[nombre] = {doc_ids[i]: s[i] for i in orden}

# Documentos a seguir: unión de los top 10 de las tres configuraciones
docs4 = []
for nombre in configs:
    for d, p in pos_cfg[nombre].items():
        if p <= 10 and d not in docs4:
            docs4.append(d)
for d in ("733", "75682"):
    if d not in docs4:
        docs4.append(d)

filas = []
for d in docs4:
    i = doc_ids.index(d)
    fila = {"doc_id": d, "Título": df_docs["titulo"][i][:35], "|d|": int(dl[i]),
            "|d|/avgdl": round(dl[i] / avgdl, 2)}
    for nombre in configs:
        fila[f"Pos {nombre}"] = pos_cfg[nombre].get(d, ">")
    for nombre in configs:
        fila[f"Score {nombre}"] = round(sc_cfg[nombre].get(d, 0.0), 3)
    filas.append(fila)

tabla4 = pd.DataFrame(filas).sort_values("Pos Base").reset_index(drop=True)
display(tabla4)

# Aportes por término de 733 y 75682 en cada configuración
en_vocab4, _ = terminos_consulta(texto4)
filas_ap = []
for d in ("733", "75682"):
    i = doc_ids.index(d)
    for t in sorted(set(en_vocab4)):
        fila = {"doc_id": d, "término": t, "f(t,d)": int(TF[i, pos[t]]),
                "techo (k1+1)·IDF base": round(idf_bm25[t] * 2.2, 3)}
        for nombre, (k1, b) in configs.items():
            fila[nombre] = round(aporte_bm25(t, d, k1, b), 4)
        filas_ap.append(fila)
display(pd.DataFrame(filas_ap))

,doc_id,Título,|d|,|d|/avgdl,Pos Base,Pos Sin longitud,Pos Más repeticiones,Score Base,Score Sin longitud,Score Más repeticiones
0,44083,The Count of the Saxon Shore; or Th,71403,1.18,1,3,1,10.840,10.886,14.437
1,34397,A Short History of Germany,33015,0.54,2,5,2,10.669,10.400,14.084
2,733,History of the Decline and Fall of,247663,4.08,3,1,3,10.606,11.041,13.966
3,47948,The archæology and prehistoric anna,299808,4.94,4,2,5,10.026,10.907,12.784
4,75682,Waterloo,27628,0.45,5,13,4,10.017,9.366,12.800
5,29143,Pius IX. And His Time,155515,2.56,6,8,7,9.341,10.185,11.757
6,44318,"The History of Rome, Books 37 to th",281228,4.63,7,7,6,9.165,10.299,11.822
7,65707,The Crime of Caste in Our Country,91299,1.50,8,11,10,9.044,9.538,11.005
8,36084,Cities of the Dawn,55866,0.92,9,19,8,8.988,8.890,11.064
9,74772,"The greatest story in the world, pe",72615,1.20,10,16,9,8.896,9.094,11.039


,doc_id,término,"f(t,d)",techo (k1+1)·IDF base,Base,Sin longitud,Más repeticiones
0,733,emperor,361,3.119,3.0851,3.1087,4.1767
1,733,legions,32,5.192,4.6193,5.0047,5.8674
2,733,roman,355,1.950,1.9288,1.9438,2.6109
3,733,war,231,0.989,0.9725,0.9841,1.3113
4,75682,emperor,114,3.119,3.0998,3.0866,4.2096
5,75682,legions,4,5.192,4.4102,3.9941,5.4651
6,75682,roman,3,1.950,1.5774,1.3931,1.9077
7,75682,war,11,0.989,0.9293,0.8919,1.2180


**Interpretación (Q7).**

- **Sin ajuste por longitud (b=0):** el denominador deja de depender de |d|, así que los documentos largos dejan de ser penalizados.
  - 733 sube del puesto 3 al 1 (10.606 → 11.041) y 47948 (4.94·avgdl) sube del 4 al 2.
  - 6049 (37 veces la media) pasa del puesto 199 al 6 (3.563 → 10.300).
  - Waterloo cae del 5 al 13 (10.017 → 9.366): en `legions` pasa de 4.4102 a 3.9941 y en `roman` de 1.5774 a 1.3931, porque pierde la ventaja de ser corto.
  - 44083 (|d|/avgdl = 1.18) casi no cambia de score (10.840 → 10.886), porque cerca de la media el factor de longitud vale casi 1. Su posición baja (1 → 3) solo porque otros suben.
- **Mayor efecto de las repeticiones (k1=2.0):** el techo por término pasa de 2.2·IDF a 3·IDF. Ganan los documentos con f alta: 733 sube de 10.606 a 13.966 (`legions` de 4.6193 a 5.8674). Los términos con f baja ganan menos (`roman` de Waterloo, con f=3, pasa de 1.5774 a 1.9077).
- **Documentos que no se mueven:** 44083, 34397 y 733 siguen en los puestos 1, 2 y 3. Todos suben de forma parecida (733 +3.36, 34397 +3.42, 44083 +3.60), así que las diferencias entre ellos se mantienen y el orden también. Waterloo solo sube del 5 al 4 porque supera a 47948 por 0.016 (12.800 frente a 12.784).
- **6049 con k1=2.0:** no se beneficia (3.563 → 3.573), porque con |d|/avgdl = 37 el denominador está dominado por el término de longitud y subir k1 afecta casi igual al numerador y al denominador.

**Conclusión:** b controla cuánto pesa la longitud y k1 controla dónde se satura la frecuencia. Cuando los candidatos ya están cerca de su techo, los scores cambian pero el orden se mantiene.